# 03  Clean Runtime & Genre


## 1. Load deduplicated data

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

info = pd.read_csv('rt_movie_info_deduped.csv')
print(f"rt_movie_info: {info.shape[0]:,} rows x {info.shape[1]} cols")
info.head()

rt_movie_info: 1,560 rows x 12 cols


,id,synopsis,rating,genre,director,writer,theater_date,dvd_date,currency,box_office,runtime,studio
0,1,"This gritty, fast-paced, and innovative police...",R,Action and Adventure|Classics|Drama,William Friedkin,Ernest Tidyman,"Oct 9, 1971","Sep 25, 2001",NaN,NaN,104 minutes,NaN
1,3,"New York City, not-too-distant-future: Eric Pa...",R,Drama|Science Fiction and Fantasy,David Cronenberg,David Cronenberg|Don DeLillo,"Aug 17, 2012","Jan 1, 2013",$,"600,000",108 minutes,Entertainment One
2,5,Illeana Douglas delivers a superb performance ...,R,Drama|Musical and Performing Arts,Allison Anders,Allison Anders,"Sep 13, 1996","Apr 18, 2000",NaN,NaN,116 minutes,NaN
3,6,Michael Douglas runs afoul of a treacherous su...,R,Drama|Mystery and Suspense,Barry Levinson,Paul Attanasio|Michael Crichton,"Dec 9, 1994","Aug 27, 1997",NaN,NaN,128 minutes,NaN
4,7,NaN,NR,Drama|Romance,Rodney Bennett,Giles Cooper,NaN,NaN,NaN,NaN,200 minutes,NaN


## 2. Clean `runtime` — strip " minutes" text, convert to numeric
Raw values look like `"104 minutes"` (a string) — we want a clean numeric `runtime_min` column so it can actually be used in calculations and Tableau.

In [2]:
info['runtime_min'] = (
    info['runtime']
    .str.replace(' minutes', '', regex=False)
    .str.strip()
)
info['runtime_min'] = pd.to_numeric(info['runtime_min'], errors='coerce')

print(f"Missing runtime (raw) before: {info['runtime'].isna().sum()}")
print(f"Missing runtime_min after conversion: {info['runtime_min'].isna().sum()}")
info[['runtime', 'runtime_min']].head()

Missing runtime (raw) before: 30
Missing runtime_min after conversion: 30


,runtime,runtime_min
0,104 minutes,104.0
1,108 minutes,108.0
2,116 minutes,116.0
3,128 minutes,128.0
4,200 minutes,200.0


In [3]:
info['runtime_min'].describe()

count    1530.000000
mean      103.967974
std        24.642392
min         5.000000
25%        91.000000
50%       100.000000
75%       114.000000
max       358.000000
Name: runtime_min, dtype: float64

## 3. Clean `genre` — pipe-delimited, multi-valued text field
Raw values look like `"Action and Adventure|Classics|Drama"` — a single string holding multiple genres. We split this out into a proper list type, plus a `genre_primary` (first-listed genre) and `genre_count` for easier filtering/grouping downstream.

In [4]:
info['genre'] = info['genre'].str.strip()

info['genre_list'] = info['genre'].apply(
    lambda x: [g.strip() for g in x.split('|')] if pd.notna(x) else []
)
info['genre_primary'] = info['genre_list'].apply(lambda g: g[0] if g else np.nan)
info['genre_count'] = info['genre_list'].apply(len)

info[['genre', 'genre_list', 'genre_primary', 'genre_count']].head()

,genre,genre_list,genre_primary,genre_count
0,Action and Adventure|Classics|Drama,"[Action and Adventure, Classics, Drama]",Action and Adventure,3
1,Drama|Science Fiction and Fantasy,"[Drama, Science Fiction and Fantasy]",Drama,2
2,Drama|Musical and Performing Arts,"[Drama, Musical and Performing Arts]",Drama,2
3,Drama|Mystery and Suspense,"[Drama, Mystery and Suspense]",Drama,2
4,Drama|Romance,"[Drama, Romance]",Drama,2


In [5]:
# Genre frequency (each movie can count toward multiple genres)
genre_exploded = info.explode('genre_list')
genre_exploded['genre_list'].value_counts()

genre_list
Drama                          912
Comedy                         550
Action and Adventure           366
Mystery and Suspense           309
Art House and International    265
Romance                        198
Classics                       193
Science Fiction and Fantasy    172
Horror                         134
Kids and Family                 99
Musical and Performing Arts     98
Documentary                     69
Special Interest                61
Western                         48
Animation                       47
Television                      23
Faith and Spirituality          11
Sports and Fitness              10
Cult Movies                      4
Anime and Manga                  2
Gay and Lesbian                  2
Name: count, dtype: int64

## 4. Quick standardization pass on other text fields

In [6]:
info['rating'] = info['rating'].str.strip()
info['studio'] = info['studio'].str.strip()
info['rating'].value_counts(dropna=False)

rating
R        521
NR       503
PG       240
PG-13    235
G         57
NaN        3
NC17       1
Name: count, dtype: int64

## 5. Finalize and export

In [7]:
# Drop the intermediate list column (not CSV-friendly) and rename id for a clearer merge key later
info_clean = info.drop(columns=['genre_list']).rename(columns={'id': 'movie_id'})
info_clean.head()

,movie_id,synopsis,rating,genre,director,writer,theater_date,dvd_date,currency,box_office,runtime,studio,runtime_min,genre_primary,genre_count
0,1,"This gritty, fast-paced, and innovative police...",R,Action and Adventure|Classics|Drama,William Friedkin,Ernest Tidyman,"Oct 9, 1971","Sep 25, 2001",NaN,NaN,104 minutes,NaN,104.0,Action and Adventure,3
1,3,"New York City, not-too-distant-future: Eric Pa...",R,Drama|Science Fiction and Fantasy,David Cronenberg,David Cronenberg|Don DeLillo,"Aug 17, 2012","Jan 1, 2013",$,"600,000",108 minutes,Entertainment One,108.0,Drama,2
2,5,Illeana Douglas delivers a superb performance ...,R,Drama|Musical and Performing Arts,Allison Anders,Allison Anders,"Sep 13, 1996","Apr 18, 2000",NaN,NaN,116 minutes,NaN,116.0,Drama,2
3,6,Michael Douglas runs afoul of a treacherous su...,R,Drama|Mystery and Suspense,Barry Levinson,Paul Attanasio|Michael Crichton,"Dec 9, 1994","Aug 27, 1997",NaN,NaN,128 minutes,NaN,128.0,Drama,2
4,7,NaN,NR,Drama|Romance,Rodney Bennett,Giles Cooper,NaN,NaN,NaN,NaN,200 minutes,NaN,200.0,Drama,2


In [8]:
info_clean.to_csv('rt_movie_info_clean.csv', index=False)
print("Exported: rt_movie_info_clean.csv")
print("Next: merge with aggregated rt_reviews (rt_reviews_deduped.csv from notebook 02)")

Exported: rt_movie_info_clean.csv
Next: merge with aggregated rt_reviews (rt_reviews_deduped.csv from notebook 02)
